# Depth-dependent centre propagation: a modal-dynamics proxy

This notebook uses the existing vertical-profile dictionary to test the kinematic prediction

\[d\mathbf{T}/dt pprox \mathbf{U}_{surface}-\mathbf{U}_{deep}.\]

It is a practical precursor to a full baroclinic-mode decomposition. It does not re-estimate `TiltDis` or `TiltDir`.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd()
ANALYSIS_ROOT = HERE.parent
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(ANALYSIS_ROOT / "beta_effect_background_flow") not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT / "beta_effect_background_flow"))

import seacofs_tilt_tools as tilt
import mechanism_tools as mech

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = mech.require_tilt_measurements(df)
print(f"Rows: {len(df):,}; measured tilts: {df.TiltDis.notna().sum():,}; eddies: {df.Eddy.nunique():,}")


Rows: 127,426; measured tilts: 105,621; eddies: 2,982


In [2]:
dic_vert = tilt.load_vert(paths, dic_form=True)
data = tilt.add_top_bottom_speeds(df, dic_vert, zmax=1000.0)
data = mech.add_tilt_components(data)

# Existing helper supplies grid-coordinate daily increments. Rotate them to geographic east/north.
alpha = grid.angle
for level in ["top", "btm"]:
    data[f"d{level}_east_km"] = data[f"dx_{level}"] * np.cos(alpha) - data[f"dy_{level}"] * np.sin(alpha)
    data[f"d{level}_north_km"] = data[f"dx_{level}"] * np.sin(alpha) + data[f"dy_{level}"] * np.cos(alpha)
data["depth_diff_east_km"] = data["dtop_east_km"] - data["dbtm_east_km"]
data["depth_diff_north_km"] = data["dtop_north_km"] - data["dbtm_north_km"]
data["depth_diff_dir"] = mech.bearing_from_east_north(data.depth_diff_east_km, data.depth_diff_north_km)
data["tilt_depth_diff_offset"] = mech.signed_angle_difference(data.TiltDir, data.depth_diff_dir)
display(mech.circular_offset_summary(data, ["tilt_depth_diff_offset"], group=("Cyc",)))


,Cyc,metric,eddies,mean_offset_deg,resultant_length,median_abs_offset_deg
0,AE,tilt_depth_diff_offset,1420,69.052897,0.289698,114.290480
1,CE,tilt_depth_diff_offset,1529,-75.101530,0.328419,248.812547


In [3]:
# Does daily differential propagation predict the daily change of measured tilt components?
data = data.sort_values(["Eddy", "Day"])
dt = data.groupby("Eddy")["Day"].diff()
for component in ["east", "north"]:
    data[f"dtilt_{component}_km_day"] = data.groupby("Eddy")[f"tilt_{component}_km"].diff() / dt

import statsmodels.formula.api as smf
rows = []
for cyc, part in data.groupby("Cyc"):
    for component in ["east", "north"]:
        use = part.dropna(subset=[f"dtilt_{component}_km_day", f"depth_diff_{component}_km"])
        fit = smf.gee(f"dtilt_{component}_km_day ~ depth_diff_{component}_km", groups="Eddy", data=use).fit()
        term = f"depth_diff_{component}_km"
        rows.append({"Cyc": cyc, "component": component, "rows": len(use),
                     "eddies": use.Eddy.nunique(), "slope": fit.params[term],
                     "ci_low": fit.conf_int().loc[term, 0], "ci_high": fit.conf_int().loc[term, 1]})
display(pd.DataFrame(rows))


,Cyc,component,rows,eddies,slope,ci_low,ci_high
0,AE,east,48568,1418,0.027317,0.018306,0.036327
1,AE,north,48568,1418,0.024364,0.014340,0.034388
2,CE,east,46447,1529,0.023085,0.014978,0.031193
3,CE,north,46447,1529,0.021235,0.013260,0.029210


In [4]:
# Maturity sensitivity: modal separation may accumulate rather than respond instantaneously.
data = tilt.add_time_coordinates(data)
data["life_stage"] = pd.cut(data.norm_time, [0, .25, .75, 1], include_lowest=True,
                            labels=["young", "mature", "decaying"])
display(mech.circular_offset_summary(data, ["tilt_depth_diff_offset"], group=("Cyc", "life_stage")))


/home/z5297792/UNSW-MRes/MRes/seacofs_eddy_tilt_analysis/tilt_mechanisms/mechanism_tools.py:145: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for keys, part in df.groupby(group, dropna=False):


,Cyc,life_stage,metric,eddies,mean_offset_deg,resultant_length,median_abs_offset_deg
0,AE,young,tilt_depth_diff_offset,1336,92.027944,0.135435,156.812133
1,AE,mature,tilt_depth_diff_offset,1413,70.394790,0.257892,121.298669
2,AE,decaying,tilt_depth_diff_offset,1320,38.271077,0.164307,151.830867
3,CE,young,tilt_depth_diff_offset,1465,-113.134373,0.147154,208.302334
4,CE,mature,tilt_depth_diff_offset,1524,-71.253795,0.299300,244.304214
5,CE,decaying,tilt_depth_diff_offset,1468,-55.411814,0.172860,219.495950


## Escalation to full modal decomposition

Proceed only if depth-dependent centre motion is coherent. Then solve vertical modes from the density profiles, estimate mode-1/mode-2 phase speeds, reconstruct their relative displacement, and test whether the reconstructed vector predicts measured tilt. If this kinematic prerequisite fails, a costly modal decomposition is unlikely to explain the observed axes.
